# Computational Theory Problems

## Problem 1: Representing SHA-256 Data

In [1]:
import numpy as np

# The biggest value a 32-bit word can hold.
w = np.uint32(0xFFFFFFFF)
print(w)
print(type(w))

4294967295
<class 'numpy.uint32'>


In [2]:
# Add 1 to the biggest 32-bit value.
with np.errstate(over="ignore"):
    print(w + np.uint32(1))

0


### 32-bit words

SHA-256 uses 32 bit numbers, so normal Python integers don't work thus I use np.uint32. This makes sure numbers don't go above the allowed limit and if a number does get too big, it automatically starts up from 0 again. NumPy warns about the overflow, so I muted it with ignore for tidiness sakes.

In [3]:
# A sequence of 32-bit words: a NumPy array of uint32.
words = np.array([1, 2, 3, 4], dtype=np.uint32)
print(words)
print(words.dtype)
print(words[0])

[1 2 3 4]
uint32
1


In [4]:
# Every item wraps on its own.
big = np.array([0xFFFFFFFF, 0xFFFFFFFE], dtype=np.uint32)
with np.errstate(over="ignore"):
    print(big + np.uint32(1))

[         0 4294967295]


### Sequences of 32-bit words

SHA-256 works on groups of numbers called words. Each block contains 16 words and the message schedule contains 64. I used a NumPy array with `np.uint32` to store them, which makes sure every number is the same size (32-bit). Each number is stored separately and if one gets too big it wraps back around to 0.

In [5]:
# Message as bytes, the text should be encoded first.
msg = "abc é".encode("utf-8")
print(msg)
print(len(msg))

b'abc \xc3\xa9'
6


### Input messages

I used bytes to store the message because SHA-256 works with data in binary form. Bytes store information in small groups of 8 bits. Once a message is stored, it cannot be changed accidentally. If I use text then I first need to convert it into bytes. Letters like 'é', take up more space than normal letters thus this is why the output of the example is 6 instead of 5.

In [6]:
# The same four bytes read in two different orders.
b = bytes([0x01, 0x02, 0x03, 0x04])
print(hex(int.from_bytes(b, "big")))
print(hex(int.from_bytes(b, "little")))

0x1020304
0x4030201


### Big-endian integers

SHA-256 reads bytes in a specific order called Big-endian. This means it reads the biggest part of the number first. I used int.from_bytes(b, "big") to make sure the bytes are read in the correct order. If I used them in the wrong order then I would get a different number, which would then give me the wrong hash.

In [7]:
# 64 bytes to sixteen 32-bit words reading in big-endian.
raw = bytes(range(64))
block = np.frombuffer(raw, dtype=">u4").astype(np.uint32)
print(block.shape)
print(hex(block[0]))

(16,)
0x10203


### 512-bit message blocks

A block contains 512 bits, which translates to 64 bytes or 16 words. I used a NumPy array to store these 16 words. np.frombuffer reads the bytes in the correct order (big-endian), and .astype(np.uint32) changes them into the format I need to work with. The first word is made from the first 4 bytes (0, 1, 2 and 3).

In [8]:
# Eight 32-bit words to 32 bytes to 64 hex characters.
# These are SHA-256 initial hash values.
H = np.array([
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19,
], dtype=np.uint32)
print(H.astype(">u4").tobytes().hex())

6a09e667bb67ae853c6ef372a54ff53a510e527f9b05688c1f83d9ab5be0cd19


### The 256-bit hash value

SHA-256 produces a 256-bit hash, which is made up of 8 smaller 32-bit numbers. I stored these in a NumPy array. In order to display the final result, I convert the numbers into bytes and then into hexadecimal, which gives me a 64-character string. The example uses the starting values that SHA-256 uses before processing a message.

### References

- [NumPy data types](https://numpy.org/doc/stable/reference/arrays.dtypes.html). For 'uint32' and the '">u4"' big-endian dtype used for blocks and the hash.
- [Python `int.from_bytes`](https://docs.python.org/3/library/stdtypes.html). The function used to read bytes in big-endian order.